# Extraction de données multimodales — flux RSS Bluesky (AFP Factuel)

Ce notebook part du flux RSS du compte Bluesky [@factuel.afp.com](https://bsky.app/profile/factuel.afp.com) pour récupérer le texte et l'image de chaque article de fact-checking mentionné.


## Sommaire
1. Configuration
2. Lecture et parsing du flux RSS
3. Récupération de l'article (texte + image)
4. Exécution de la collecte
5. Téléchargement des images
6. Sauvegarde du dataset consolidé
7. Exploration rapide

## 1. Configuration

In [19]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin
from xml.etree import ElementTree

import pandas as pd
import requests
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests
from tqdm.auto import tqdm

In [20]:
RSS_URL = "https://bsky.app/profile/did:plc:4ks5wkubjfcbgxvphqkd3wxm/rss"

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "bluesky_afp"
IMAGES_DIR = DATA_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; Projet12-FactCheckCollector/1.0)"}
REQUEST_DELAY_SECONDS = 1.0
MIN_ARTICLE_TEXT_LENGTH = 200

# u.afp.com / factuel.afp.com sit behind an Akamai WAF that blocks plain `requests`
# calls based on TLS fingerprint (not just headers) - curl_cffi impersonates a real
# Chrome TLS handshake, which gets through where `requests` gets a 403.
BROWSER_IMPERSONATION = "chrome124"

## 2. Lecture et parsing du flux RSS

Le lien vers l'article complet n'est pas un champ RSS dédié : c'est la dernière URL trouvée dans le texte du post (les posts AFP se terminent presque toujours par le lien de décryptage).

In [21]:
def parse_rss_feed(feed_url: str) -> pd.DataFrame:
    """Fetch and parse a Bluesky RSS feed into one row per post."""
    response = requests.get(feed_url, headers=HTTP_HEADERS, timeout=30)
    response.raise_for_status()

    root = ElementTree.fromstring(response.content)
    rows = []
    for item in root.iter("item"):
        post_text = (item.findtext("description") or "").strip()
        urls_in_text = re.findall(r"https?://\S+", post_text)

        rows.append(
            {
                "post_url": item.findtext("link"),
                "post_text": post_text,
                "pub_date": item.findtext("pubDate"),
                "guid": item.findtext("guid"),
                # The article link is usually the last URL mentioned in the post text.
                "article_url": urls_in_text[-1] if urls_in_text else None,
            }
        )

    df = pd.DataFrame(rows)
    df["pub_date"] = pd.to_datetime(df["pub_date"], utc=True, errors="coerce")
    return df

In [22]:
df = parse_rss_feed(RSS_URL)
print(f"Parsed {len(df)} post(s), {df['article_url'].notna().sum()} with a detected article link")
df.head()

Parsed 30 post(s), 28 with a detected article link


,post_url,post_text,pub_date,guid,article_url
0,https://bsky.app/profile/factuel.afp.com/post/...,📸 Une photo de François Mitterrand dînant ave...,2026-09-15 12:41:00+00:00,at://did:plc:4ks5wkubjfcbgxvphqkd3wxm/app.bsky...,https://u.afp.com/Sj84
1,https://bsky.app/profile/factuel.afp.com/post/...,🍪 Des biscuits contenant du plastique bientôt ...,2026-09-14 12:06:00+00:00,at://did:plc:4ks5wkubjfcbgxvphqkd3wxm/app.bsky...,https://u.afp.com/SjeA
2,https://bsky.app/profile/factuel.afp.com/post/...,"📅 25 ans après le 11-Septembre, les fausses af...",2026-09-11 13:32:00+00:00,at://did:plc:4ks5wkubjfcbgxvphqkd3wxm/app.bsky...,https://u.afp.com/SjMK
3,https://bsky.app/profile/factuel.afp.com/post/...,🇨🇩 🇺🇳 L'ancien président congolais Joseph Kabi...,2026-09-10 16:30:00+00:00,at://did:plc:4ks5wkubjfcbgxvphqkd3wxm/app.bsky...,https://u.afp.com/Sjfo
4,https://bsky.app/profile/factuel.afp.com/post/...,🐴 🇳🇵 Une vidéo montrant le sauvetage d'un chev...,2026-09-10 13:37:00+00:00,at://did:plc:4ks5wkubjfcbgxvphqkd3wxm/app.bsky...,https://u.afp.com/Sjf4


## 3. Récupération de l'article (texte + image)

Une seule requête par article : texte principal (mêmes heuristiques que pour les autres sources du projet) et image principale (`og:image` / `twitter:image`).

In [23]:
def fetch_article(url, min_length: int = MIN_ARTICLE_TEXT_LENGTH) -> tuple[int | None, str | None, str | None]:
    """Fetch an article page. Returns (status_code, article_text, image_url).

    Uses curl_cffi (Chrome TLS impersonation) rather than plain `requests`: AFP's
    WAF blocks the latter by TLS fingerprint regardless of headers used.

    status_code is None on a network-level failure (timeout, DNS failure, WAF block
    before any response, ...). article_text is None if the page isn't a usable 200,
    or its extracted text is too short. image_url is None if no og:image/twitter:image
    meta tag is found.
    """
    try:
        response = curl_requests.get(
            url, headers=HTTP_HEADERS, timeout=15, allow_redirects=True, impersonate=BROWSER_IMPERSONATION
        )
    except curl_requests.errors.RequestsError:
        return None, None, None

    if response.status_code != 200:
        return response.status_code, None, None

    soup = BeautifulSoup(response.text, "html.parser")

    image_url = None
    for meta_name in ("og:image", "twitter:image", "twitter:image:src"):
        tag = soup.find("meta", attrs={"property": meta_name}) or soup.find("meta", attrs={"name": meta_name})
        if tag and tag.get("content"):
            image_url = urljoin(response.url, tag["content"])
            break

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    article = soup.find("article") or soup.find("main") or soup.find(attrs={"role": "main"})
    paragraphs = (article or soup).find_all("p")

    text = "\n".join(p.get_text(" ", strip=True) for p in paragraphs)
    text = re.sub(r"\n{2,}", "\n", text).strip()

    return response.status_code, (text if len(text) >= min_length else None), image_url

## 4. Exécution de la collecte

In [24]:
statuses, texts, image_urls = [], [], []

for article_url in tqdm(df["article_url"], desc="Fetching articles"):
    if pd.isna(article_url):
        statuses.append(None)
        texts.append(None)
        image_urls.append(None)
        continue

    status, text, image_url = fetch_article(article_url)
    statuses.append(status)
    texts.append(text)
    image_urls.append(image_url)
    time.sleep(REQUEST_DELAY_SECONDS)

df["http_status"] = statuses
df["article_text"] = texts
df["image_url"] = image_urls

print(f"{df['http_status'].eq(200).sum()}/{len(df)} article(s) fetched successfully")
print(f"{df['article_text'].notna().sum()} with usable text, {df['image_url'].notna().sum()} with an image")
df["http_status"].value_counts(dropna=False).sort_index()

Fetching articles:   0%|          | 0/30 [00:00<?, ?it/s]

27/30 article(s) fetched successfully
27 with usable text, 27 with an image


http_status
200.0    27
403.0     1
NaN       2
Name: count, dtype: int64

## 5. Téléchargement des images

In [25]:
def download_image(image_url: str, dest_path: Path) -> bool:
    """Download a single image to dest_path. Returns True on success."""
    try:
        response = curl_requests.get(image_url, headers=HTTP_HEADERS, timeout=15, impersonate=BROWSER_IMPERSONATION)
        response.raise_for_status()
        dest_path.write_bytes(response.content)
        return True
    except curl_requests.errors.RequestsError:
        return False

In [26]:
local_paths = []
for idx, image_url in tqdm(list(enumerate(df["image_url"])), desc="Downloading images"):
    if pd.isna(image_url):
        local_paths.append(None)
        continue

    extension = Path(image_url.split("?")[0]).suffix or ".jpg"
    dest_path = IMAGES_DIR / f"post_{idx:05d}{extension}"

    if download_image(image_url, dest_path):
        local_paths.append(str(dest_path.relative_to(PROJECT_ROOT)))
    else:
        local_paths.append(None)
    time.sleep(REQUEST_DELAY_SECONDS)

df["local_image_path"] = local_paths
print(f"{df['local_image_path'].notna().sum()} image(s) downloaded")

27 image(s) downloaded


## 6. Sauvegarde du dataset consolidé

In [27]:
output_csv = DATA_DIR / "bluesky_afp_multimodal.csv"
df.to_csv(output_csv, index=False)
print(f"Saved {len(df)} row(s) to {output_csv}")

Saved 30 row(s) to c:\Formation\Projet_12\data\bluesky_afp\bluesky_afp_multimodal.csv


## 7. Exploration rapide

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   post_url          30 non-null     str                
 1   post_text         30 non-null     str                
 2   pub_date          30 non-null     datetime64[us, UTC]
 3   guid              30 non-null     str                
 4   article_url       28 non-null     str                
 5   http_status       28 non-null     float64            
 6   article_text      27 non-null     str                
 7   image_url         27 non-null     str                
 8   local_image_path  27 non-null     str                
dtypes: datetime64[us, UTC](1), float64(1), str(7)
memory usage: 2.2 KB


In [29]:
df[df["local_image_path"].notna()][
    ["post_text", "article_url", "article_text", "local_image_path"]
].head(5)

,post_text,article_url,article_text,local_image_path
0,📸 Une photo de François Mitterrand dînant ave...,https://u.afp.com/Sj84,De fausses photos historiques circulent réguli...,data\bluesky_afp\images\post_00000.jpeg
1,🍪 Des biscuits contenant du plastique bientôt ...,https://u.afp.com/SjeA,"Le 24 août 2026, une équipe de chercheurs a pr...",data\bluesky_afp\images\post_00001.jpeg
2,"📅 25 ans après le 11-Septembre, les fausses af...",https://u.afp.com/SjMK,"Le 11 septembre 2001, des avions détournés par...",data\bluesky_afp\images\post_00002.jpeg
3,🇨🇩 🇺🇳 L'ancien président congolais Joseph Kabi...,https://u.afp.com/Sjfo,Le mandat de l’actuel secrétaire général de l’...,data\bluesky_afp\images\post_00003.jpeg
4,🐴 🇳🇵 Une vidéo montrant le sauvetage d'un chev...,https://u.afp.com/Sjf4,"Le 26 août, l'effondrement d'un glacier dans l...",data\bluesky_afp\images\post_00004.jpeg
